# ЛР5 — Семантический анализатор + генерация текста (LSTM, seq2seq)

Продолжение ЛР4: к трём анализаторам тональности добавляется четвёртый нейросетевой (LSTM) и генератор осмысленного текста, работающий только на основе семантических признаков датасета.
- A) ML.NET FeaturizeText + SDCA MaximumEntropy
- B) ML.NET FeaturizeText + L-BFGS MaximumEntropy
- C) CNN (ONNX Runtime, обучение в Python, инференс в C#)
- D) LSTM-классификатор (PyTorch → ONNX → C#)
- G) LSTM seq2seq-генератор текста (PyTorch → ONNX → C#)

Генератор использует только семантику из датасета (sentiment, время твита, возраст, страна) и обученный LSTM-язык, без внешних языковых моделей.


In [1]:
#r "nuget: Microsoft.ML, 3.0.1"
#r "nuget: Microsoft.ML.FastTree, 3.0.1"
#r "nuget: Microsoft.ML.OnnxRuntime, 1.19.0"
#r "nuget: CsvHelper, 30.0.1"


The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages CsvHelper, 30.0.1 Microsoft.ML, 3.0.1 Microsoft.ML.FastTree, 3.0.1 Microsoft.ML.OnnxRuntime, 1.19.0

In [2]:
using System;
using System.IO;
using System.Linq;
using System.Text;
using System.Text.RegularExpressions;
using System.Collections.Generic;
using System.Globalization;
using CsvHelper;
using CsvHelper.Configuration;
using CsvHelper.Configuration.Attributes;
using Microsoft.ML;
using Microsoft.ML.Data;
using Microsoft.ML.Transforms.Text;
using Microsoft.ML.Trainers;
using Microsoft.ML.OnnxRuntime;
using Microsoft.ML.OnnxRuntime.Tensors;


In [3]:
public class TrainRow { public string textID {get;set;} public string text {get;set;} public string selected_text {get;set;} public string sentiment {get;set;} [Name("Time of Tweet")] public string TimeOfTweet {get;set;} [Name("Age of User")] public string AgeOfUser {get;set;} public string Country {get;set;} }
public class TestRow  { public string textID {get;set;} public string text {get;set;} public string sentiment {get;set;} [Name("Time of Tweet")] public string TimeOfTweet {get;set;} [Name("Age of User")] public string AgeOfUser {get;set;} public string Country {get;set;} }
public class SentimentInput { [LoadColumn(0)] public string Text {get;set;} [LoadColumn(1)] public string Sentiment {get;set;} }
public class SentimentPrediction { [ColumnName("PredictedLabel")] public string PredictedLabel {get;set;} }

static string Normalize(string s){ if (string.IsNullOrWhiteSpace(s)) return string.Empty; s=s.ToLowerInvariant(); s=Regex.Replace(s, "https?://\\S+", " " ); s=Regex.Replace(s, "[\\\"'`]+", "" ); s=Regex.Replace(s, "[^\\p{L}\\p{Nd}#@ ]+", " " ); s=Regex.Replace(s, "\\s+", " " ).Trim(); return s; }

static List<SentimentInput> LoadTrain(string path){ using var sr=new StreamReader(path, Encoding.UTF8); var cfg=new CsvConfiguration(CultureInfo.InvariantCulture){HasHeaderRecord=true, DetectDelimiter=true, IgnoreBlankLines=true, BadDataFound=null}; using var csv=new CsvReader(sr,cfg); var list=new List<SentimentInput>(); foreach (var r in csv.GetRecords<TrainRow>()){ if (string.IsNullOrWhiteSpace(r.text) || string.IsNullOrWhiteSpace(r.sentiment)) continue; list.Add(new SentimentInput{ Text=Normalize(r.text), Sentiment=r.sentiment.Trim()}); } return list; }
static List<SentimentInput> LoadTest(string path){ using var sr=new StreamReader(path, Encoding.UTF8); var cfg=new CsvConfiguration(CultureInfo.InvariantCulture){HasHeaderRecord=true, DetectDelimiter=true, IgnoreBlankLines=true, BadDataFound=null}; using var csv=new CsvReader(sr,cfg); var list=new List<SentimentInput>(); foreach (var r in csv.GetRecords<TestRow>()){ if (string.IsNullOrWhiteSpace(r.text) || string.IsNullOrWhiteSpace(r.sentiment)) continue; list.Add(new SentimentInput{ Text=Normalize(r.text), Sentiment=r.sentiment.Trim()}); } return list; }

var dataDir = Path.Combine("..", "data");
var trainPath = Path.Combine(dataDir, "train.csv");
var testPath  = Path.Combine(dataDir, "test.csv");
var train = LoadTrain(trainPath); var test = LoadTest(testPath);
Console.WriteLine($"Train={train.Count}, Test={test.Count}");


Train=27480, Test=3534


In [4]:
var ml = new MLContext(seed:42);
var trainDV = ml.Data.LoadFromEnumerable(train);
var testDV  = ml.Data.LoadFromEnumerable(test);

IEstimator<ITransformer> Pipeline(string trainer){
    var text = ml.Transforms.Text.FeaturizeText("Features", nameof(SentimentInput.Text));
    var key = ml.Transforms.Conversion.MapValueToKey("Label", nameof(SentimentInput.Sentiment));
    IEstimator<ITransformer> tr = trainer=="sdca" ? ml.MulticlassClassification.Trainers.SdcaMaximumEntropy(labelColumnName:"Label", featureColumnName:"Features") : ml.MulticlassClassification.Trainers.LbfgsMaximumEntropy(labelColumnName:"Label", featureColumnName:"Features");
    return text.Append(key).Append(tr).Append(ml.Transforms.Conversion.MapKeyToValue("PredictedLabel"));
}

(ITransformer model, string name) TrainEval(string name, string trainer){
    var pipe = Pipeline(trainer);
    var model = pipe.Fit(trainDV);
    var preds = model.Transform(testDV);
    var metrics = ml.MulticlassClassification.Evaluate(preds, labelColumnName:"Label", predictedLabelColumnName:"PredictedLabel");
    Console.WriteLine($"[{name}] MicroAcc={metrics.MicroAccuracy:F3}  MacroAcc={metrics.MacroAccuracy:F3}");
    return (model,name);
}

var (modelA, nameA) = TrainEval("A: FeaturizeText + SDCA", "sdca");
var (modelB, nameB) = TrainEval("B: FeaturizeText + L-BFGS", "lbfgs");


[A: FeaturizeText + SDCA] MicroAcc=0,699  MacroAcc=0,693


[B: FeaturizeText + L-BFGS] MicroAcc=0,699  MacroAcc=0,692


In [5]:
string modelsDir = Path.Combine("..", "models");
string onnxPath  = Path.Combine(modelsDir, "cnn_text.onnx");
string vocabPath = Path.Combine(modelsDir, "vocab.json");
string cfgPath   = Path.Combine(modelsDir, "config.json");

Dictionary<string,int>? vocab = null; int maxLen = 40;
if (File.Exists(vocabPath)) { var json = System.Text.Json.JsonDocument.Parse(File.ReadAllText(vocabPath)); vocab = new Dictionary<string,int>(); foreach (var kv in json.RootElement.EnumerateObject()) vocab[kv.Name]=kv.Value.GetInt32(); }
if (File.Exists(cfgPath)) { var json = System.Text.Json.JsonDocument.Parse(File.ReadAllText(cfgPath)); if (json.RootElement.TryGetProperty("max_len", out var v)) maxLen = v.GetInt32(); }

int Id(string tok){ if (vocab==null) return 1; return vocab.TryGetValue(tok, out var id) ? id : 1; } // 0=pad, 1=unk
int[] ToIds(string text){ var toks = Regex.Split(Normalize(text), "\\s+").Where(t=>t.Length>0).ToArray(); var arr = new int[maxLen]; for (int i=0;i<Math.Min(maxLen,toks.Length);i++) arr[i]=Id(toks[i]); return arr; }

if (File.Exists(onnxPath) && vocab!=null) {
    using var session = new InferenceSession(onnxPath, new SessionOptions());
    double SoftmaxMax(double[] z, out int arg){ double m=z.Max(); double s=0; var p=new double[z.Length]; for(int i=0;i<z.Length;i++){ p[i]=Math.Exp(z[i]-m); s+=p[i]; } double best=-1; arg=0; for(int i=0;i<z.Length;i++){ var v=p[i]/s; if(v>best){best=v; arg=i;} } return p[arg]/s; }
    int correct=0; int total=0; foreach(var ex in test){ var ids = ToIds(ex.Text); var t = new DenseTensor<long>(new[] {1, maxLen}); for(int i=0;i<maxLen;i++) t[0,i] = ids[i]; var inputs = new List<NamedOnnxValue>{ NamedOnnxValue.CreateFromTensor("input_ids", t) }; using var results = session.Run(inputs); var logits = results.First().AsEnumerable<float>().ToArray(); int arg; var _ = SoftmaxMax(Array.ConvertAll(logits, x=>(double)x), out arg); string pred = arg==0?"negative": arg==1?"neutral":"positive"; if (pred.Equals(ex.Sentiment, StringComparison.OrdinalIgnoreCase)) correct++; total++; }
    Console.WriteLine($"[C: CNN-ONNX] Acc={correct/(double)total:F3} (total={total})");
} else {
    Console.WriteLine("[C: CNN-ONNX] Модель не найдена. Запустите src/train_cnn.py или src/train_cnn.ipynb для обучения и экспорта.");
}


[C: CNN-ONNX] Acc=0,658 (total=3534)



(6,23): warning CS8632: Аннотацию для ссылочных типов, допускающих значения NULL, следует использовать в коде только в контексте аннотаций "#nullable".



<null>

In [6]:
string lstmModelsDir = Path.Combine("..", "models");
string lstmOnnxPath  = Path.Combine(lstmModelsDir, "lstm_cls.onnx");
string lstmVocabPath = Path.Combine(lstmModelsDir, "lstm_cls_vocab.json");
string lstmCfgPath   = Path.Combine(lstmModelsDir, "lstm_cls_config.json");

Dictionary<string,int>? lstmVocab = null; int lstmMaxLen = 40;
if (File.Exists(lstmVocabPath)) { var json = System.Text.Json.JsonDocument.Parse(File.ReadAllText(lstmVocabPath)); lstmVocab = new Dictionary<string,int>(); foreach (var kv in json.RootElement.EnumerateObject()) lstmVocab[kv.Name]=kv.Value.GetInt32(); }
if (File.Exists(lstmCfgPath)) { var json = System.Text.Json.JsonDocument.Parse(File.ReadAllText(lstmCfgPath)); if (json.RootElement.TryGetProperty("max_len", out var v)) lstmMaxLen = v.GetInt32(); }

int LstmId(string tok){ if (lstmVocab==null) return 1; return lstmVocab.TryGetValue(tok, out var id) ? id : 1; } // 0=pad, 1=unk
int[] LstmToIds(string text){ var toks = Regex.Split(Normalize(text), "\\s+").Where(t=>t.Length>0).ToArray(); var arr = new int[lstmMaxLen]; for (int i=0;i<Math.Min(lstmMaxLen,toks.Length);i++) arr[i]=LstmId(toks[i]); return arr; }

if (File.Exists(lstmOnnxPath) && lstmVocab!=null) {
    using var session = new InferenceSession(lstmOnnxPath, new SessionOptions());
    double SoftmaxMax(double[] z, out int arg){ double m=z.Max(); double s=0; var p=new double[z.Length]; for(int i=0;i<z.Length;i++){ p[i]=Math.Exp(z[i]-m); s+=p[i]; } double best=-1; arg=0; for(int i=0;i<z.Length;i++){ var v=p[i]/s; if(v>best){best=v; arg=i;} } return p[arg]/s; }
    int correct=0; int total=0; foreach(var ex in test){ var ids = LstmToIds(ex.Text); var t = new DenseTensor<long>(new[] {1, lstmMaxLen}); for(int i=0;i<lstmMaxLen;i++) t[0,i] = ids[i]; var inputs = new List<NamedOnnxValue>{ NamedOnnxValue.CreateFromTensor("input_ids", t) }; using var results = session.Run(inputs); var logits = results.First().AsEnumerable<float>().ToArray(); int arg; var _ = SoftmaxMax(Array.ConvertAll(logits, x=>(double)x), out arg); string pred = arg==0?"negative": arg==1?"neutral":"positive"; if (pred.Equals(ex.Sentiment, StringComparison.OrdinalIgnoreCase)) correct++; total++; }
    Console.WriteLine($"[D: LSTM-CLS-ONNX] Acc={correct/(double)total:F3} (total={total})");
} else {
    Console.WriteLine("[D: LSTM-CLS-ONNX] Модель не найдена. Запустите src/train_lstm_cls.py для обучения и экспорта.");
}


[D: LSTM-CLS-ONNX] Acc=0,427 (total=3534)



(6,23): warning CS8632: Аннотацию для ссылочных типов, допускающих значения NULL, следует использовать в коде только в контексте аннотаций "#nullable".



<null>

In [7]:
var samples = new[]{
    new { Text = "Absolutely love the latest update, everything works flawlessly and makes me so happy!", Time = "morning", Age = "0-20", Country = "Afghanistan" },
    new { Text = "Customer support was terrible: half of my order was missing and nobody apologized.", Time = "noon", Age = "21-30", Country = "Albania" },
    new { Text = "It's fine I guess overall, not amazing but acceptable for everyday use.", Time = "night", Age = "31-45", Country = "Algeria" }
};

string PredictWith(ITransformer model, string text){
    var engine = ml.Model.CreatePredictionEngine<SentimentInput, SentimentPrediction>(model);
    var input = new SentimentInput{ Text=Normalize(text), Sentiment="neutral" };
    var pred = engine.Predict(input);
    return pred.PredictedLabel;
}

string PredictCnn(string text){
    string modelsDirLocal = Path.Combine("..", "models");
    string onnxPathLocal  = Path.Combine(modelsDirLocal, "cnn_text.onnx");
    string vocabPathLocal = Path.Combine(modelsDirLocal, "vocab.json");
    string cfgPathLocal   = Path.Combine(modelsDirLocal, "config.json");
    if (!File.Exists(onnxPathLocal) || !File.Exists(vocabPathLocal)) return "n/a";
    var vocab = new Dictionary<string,int>();
    var jsonV = System.Text.Json.JsonDocument.Parse(File.ReadAllText(vocabPathLocal));
    foreach (var kv in jsonV.RootElement.EnumerateObject()) vocab[kv.Name]=kv.Value.GetInt32();
    int maxLenLocal = 40;
    if (File.Exists(cfgPathLocal)) { var jsonCfg = System.Text.Json.JsonDocument.Parse(File.ReadAllText(cfgPathLocal)); if (jsonCfg.RootElement.TryGetProperty("max_len", out var v)) maxLenLocal = v.GetInt32(); }
    int IdLocal(string tok){ return vocab.TryGetValue(tok, out var id) ? id : 1; }
    int[] ToIdsLocal(string txt){ var toks = Regex.Split(Normalize(txt), "\\s+").Where(t=>t.Length>0).ToArray(); var arr = new int[maxLenLocal]; for(int i=0;i<Math.Min(maxLenLocal,toks.Length);i++) arr[i]=IdLocal(toks[i]); return arr; }
    using var session = new InferenceSession(onnxPathLocal, new SessionOptions());
    double SoftmaxMaxLocal(double[] z, out int arg){ double m=z.Max(); double s=0; var p=new double[z.Length]; for(int i=0;i<z.Length;i++){ p[i]=Math.Exp(z[i]-m); s+=p[i]; } double best=-1; arg=0; for(int i=0;i<z.Length;i++){ var v=p[i]/s; if(v>best){best=v; arg=i;} } return p[arg]/s; }
    var ids = ToIdsLocal(text); var t = new DenseTensor<long>(new[] {1, maxLenLocal}); for(int i=0;i<maxLenLocal;i++) t[0,i]=ids[i];
    var inputs = new List<NamedOnnxValue>{ NamedOnnxValue.CreateFromTensor("input_ids", t) };
    using var results = session.Run(inputs);
    var logits = results.First().AsEnumerable<float>().ToArray(); int arg; var _ = SoftmaxMaxLocal(Array.ConvertAll(logits, x=>(double)x), out arg);
    return arg==0?"negative": arg==1?"neutral":"positive";
}

string PredictLstm(string text){
    string modelsDirLocal = Path.Combine("..", "models");
    string onnxPathLocal  = Path.Combine(modelsDirLocal, "lstm_cls.onnx");
    string vocabPathLocal = Path.Combine(modelsDirLocal, "lstm_cls_vocab.json");
    string cfgPathLocal   = Path.Combine(modelsDirLocal, "lstm_cls_config.json");
    if (!File.Exists(onnxPathLocal) || !File.Exists(vocabPathLocal)) return "n/a";
    var vocab = new Dictionary<string,int>();
    var jsonV = System.Text.Json.JsonDocument.Parse(File.ReadAllText(vocabPathLocal));
    foreach (var kv in jsonV.RootElement.EnumerateObject()) vocab[kv.Name]=kv.Value.GetInt32();
    int maxLenLocal = 40;
    if (File.Exists(cfgPathLocal)) { var jsonCfg = System.Text.Json.JsonDocument.Parse(File.ReadAllText(cfgPathLocal)); if (jsonCfg.RootElement.TryGetProperty("max_len", out var v)) maxLenLocal = v.GetInt32(); }
    int IdLocal(string tok){ return vocab.TryGetValue(tok, out var id) ? id : 1; }
    int[] ToIdsLocal(string txt){ var toks = Regex.Split(Normalize(txt), "\\s+").Where(t=>t.Length>0).ToArray(); var arr = new int[maxLenLocal]; for(int i=0;i<Math.Min(maxLenLocal,toks.Length);i++) arr[i]=IdLocal(toks[i]); return arr; }
    using var session = new InferenceSession(onnxPathLocal, new SessionOptions());
    double SoftmaxMaxLocal(double[] z, out int arg){ double m=z.Max(); double s=0; var p=new double[z.Length]; for(int i=0;i<z.Length;i++){ p[i]=Math.Exp(z[i]-m); s+=p[i]; } double best=-1; arg=0; for(int i=0;i<z.Length;i++){ var v=p[i]/s; if(v>best){best=v; arg=i;} } return p[arg]/s; }
    var ids = ToIdsLocal(text); var t = new DenseTensor<long>(new[] {1, maxLenLocal}); for(int i=0;i<maxLenLocal;i++) t[0,i]=ids[i];
    var inputs = new List<NamedOnnxValue>{ NamedOnnxValue.CreateFromTensor("input_ids", t) };
    using var results = session.Run(inputs);
    var logits = results.First().AsEnumerable<float>().ToArray(); int arg; var _ = SoftmaxMaxLocal(Array.ConvertAll(logits, x=>(double)x), out arg);
    return arg==0?"negative": arg==1?"neutral":"positive";
}

string GenerateTextFromSemantics(string sentiment, string time, string age, string country){
    string modelsDirLocal = Path.Combine("..", "models");
    string onnxPathLocal  = Path.Combine(modelsDirLocal, "seq2seq_lstm.onnx");
    string vocabPathLocal = Path.Combine(modelsDirLocal, "seq2seq_lstm_vocab.json");
    string cfgPathLocal   = Path.Combine(modelsDirLocal, "seq2seq_lstm_config.json");
    if (!File.Exists(onnxPathLocal) || !File.Exists(vocabPathLocal) || !File.Exists(cfgPathLocal)) return "Генератор не обучен (нет seq2seq_lstm.*)";

    var vocab = new Dictionary<string,int>();
    var jsonV = System.Text.Json.JsonDocument.Parse(File.ReadAllText(vocabPathLocal));
    foreach (var kv in jsonV.RootElement.EnumerateObject()) vocab[kv.Name]=kv.Value.GetInt32();
    var idToToken = new Dictionary<int,string>(); foreach (var kv in vocab) idToToken[kv.Value]=kv.Key;

    int maxLenLocal = 40; int padId = 0; int bosId = 2; int eosId = 3;
    var jsonCfg = System.Text.Json.JsonDocument.Parse(File.ReadAllText(cfgPathLocal));
    var root = jsonCfg.RootElement;
    if (root.TryGetProperty("max_len", out var vMax)) maxLenLocal = vMax.GetInt32();
    if (root.TryGetProperty("pad_id", out var vPad)) padId = vPad.GetInt32();
    if (root.TryGetProperty("bos_id", out var vBos)) bosId = vBos.GetInt32();
    if (root.TryGetProperty("eos_id", out var vEos)) eosId = vEos.GetInt32();

    int unkId = vocab.TryGetValue("<unk>", out var u) ? u : 1;
    int IdTok(string tok){ return vocab.TryGetValue(tok, out var id) ? id : unkId; }

    string CountryToken(string c){ var s = c.ToLowerInvariant(); s = Regex.Replace(s, "[^a-z]+", "_"); s = s.Trim('_'); return "country_" + s; }

    var prefixTokens = new List<string>{ "<bos>", $"sent_{sentiment.ToLowerInvariant()}", $"time_{time.ToLowerInvariant()}", $"age_{age}", CountryToken(country) };
    var ids = new List<int>(); foreach (var t in prefixTokens) ids.Add(IdTok(t));

    using var session = new InferenceSession(onnxPathLocal, new SessionOptions());
    int vocabSize = vocab.Count;
    int maxSteps = 20;

    for (int step = 0; step < maxSteps && ids.Count < maxLenLocal; step++){
        var tensor = new DenseTensor<long>(new[] {1, maxLenLocal});
        for (int i=0;i<maxLenLocal;i++) tensor[0,i] = i<ids.Count ? ids[i] : padId;
        var inputs = new List<NamedOnnxValue>{ NamedOnnxValue.CreateFromTensor("input_ids", tensor) };
        using var results = session.Run(inputs);
        var logits = results.First().AsEnumerable<float>().ToArray();
        int lastIdx = Math.Min(ids.Count, maxLenLocal) - 1; if (lastIdx < 0) lastIdx = 0;
        int offset = lastIdx * vocabSize;
        int nextId = 0; float best = float.NegativeInfinity;
        for (int i=0;i<vocabSize;i++){
            float val = logits[offset + i];
            if (val > best){ best = val; nextId = i; }
        }
        if (nextId==eosId || nextId==padId) break;
        ids.Add(nextId);
    }

    var textTokens = new List<string>();
    int skip = prefixTokens.Count;
    for (int i=skip; i<ids.Count; i++){
        if (!idToToken.TryGetValue(ids[i], out var tok)) continue;
        if (tok=="<eos>" || tok=="<pad>") break;
        if (tok.StartsWith("sent_") || tok.StartsWith("time_") || tok.StartsWith("age_") || tok.StartsWith("country_")) continue;
        textTokens.Add(tok);
    }
    return textTokens.Count>0 ? string.Join(" ", textTokens) : "(генератор не смог построить текст)";
}

foreach (var s in samples){
    var pa = PredictWith(modelA, s.Text);
    var pb = PredictWith(modelB, s.Text);
    var pc = PredictCnn(s.Text);
    var pd = PredictLstm(s.Text);
    var sentimentForGen = (pd=="negative" || pd=="neutral" || pd=="positive") ? pd : "neutral";
    var gen = GenerateTextFromSemantics(sentimentForGen, s.Time, s.Age, s.Country);
    Console.WriteLine("==== SAMPLE ====");
    Console.WriteLine($"Text: {s.Text}");
    Console.WriteLine($" A (ML.NET SDCA): {pa}");
    Console.WriteLine($" B (ML.NET L-BFGS): {pb}");
    Console.WriteLine($" C (CNN-ONNX): {pc}");
    Console.WriteLine($" D (LSTM-CLS-ONNX): {pd}");
    Console.WriteLine($" Generated (D + meta): {gen}");
    Console.WriteLine($" Meta: sentiment={sentimentForGen}, time={s.Time}, age={s.Age}, country={s.Country}");
    Console.WriteLine();
}


==== SAMPLE ====


Text: Absolutely love the latest update, everything works flawlessly and makes me so happy!


 A (ML.NET SDCA): positive


 B (ML.NET L-BFGS): positive


 C (CNN-ONNX): positive


...

 B (ML.NET L-BFGS): neutral


 C (CNN-ONNX): neutral


 D (LSTM-CLS-ONNX): neutral


 Generated (D + meta): i have to go to the new time


 Meta: sentiment=neutral, time=night, age=31-45, country=Algeria
